<a href="https://colab.research.google.com/github/Lobnaait/SEARCH_Lobna_Tsetline_CMRI/blob/main/Tsetline_ImagesDataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!pip install pyTsetlinMachine

In [12]:
#IMPORT LIBRARIES AND DATASET
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from pyTsetlinMachine.tm import MultiClassTsetlinMachine

dataset = load_digits()  # Load the images dataset

In [13]:
# Divide the dataset into training set and test set for features (X) and target (y)
from sklearn.model_selection import train_test_split
X = dataset.images # input images
y = dataset.target # output labels
X_train_full, X_test, y_train_full, y_test = train_test_split(dataset.images, dataset.target, test_size = 0.25)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.20)


In [14]:
# CHOOSE THE TRESHOLDS: Based on the number of treshold divide the dataset into tresholds
def compute_thresholds(X, n_thresholds):
  # Extract non-zero pixel
  non_zero_pixels = X[X > 0]
  percentile_values = np.linspace(0, 100, int(n_thresholds) + 2)[1:-1] # Exclude 0 and 100
  thresholds = np.percentile(non_zero_pixels, percentile_values)
  return np.unique(thresholds)

In [15]:
def booleanise(X, thresholds):
  # Flatten each image
  X_flat = X.reshape(X.shape[0], -1)
  X_bool = (X_flat[:, :, None] > thresholds[None, None, :]) #Compare every pixel with every threshold
  #convert to binary
  X_bool = X_bool.astype(np.uint32)
  X_bool = X_bool.reshape(X_bool.shape[0], -1)
  return X_bool

In [17]:
# VALIDATION: Test how many tresholds are optimal to use?
threshold_candidates = np.linspace(1,10,10)
results = []

for n_thresholds in threshold_candidates:

    thresholds = compute_thresholds(X_train, n_thresholds)
    X_train_bool = booleanise(X_train, thresholds)
    X_val_bool = booleanise(X_val, thresholds)

    # model
    tm = MultiClassTsetlinMachine(number_of_clauses=1000, T=50, s=5.0)
    tm.fit(X_train_bool, y_train, epochs=100)

    val_accuracy = accuracy_score(y_val, tm.predict(X_val_bool))

    results.append([n_thresholds, val_accuracy])

In [20]:
results = np.array(results)
print(results)


[[ 1.          0.94444444]
 [ 2.          0.97037037]
 [ 3.          0.95185185]
 [ 4.          0.96666667]
 [ 5.          0.97037037]
 [ 6.          0.96666667]
 [ 7.          0.97037037]
 [ 8.          0.95925926]
 [ 9.          0.97407407]
 [10.          0.97407407]]


In [21]:
index = np.argmax(results[:,1]) #index of max accuracy
row_with_max_accuracy = results[index]
print(f"Row with maximum accuracy score: {row_with_max_accuracy}")

Row with maximum accuracy score: [9.         0.97407407]


In [22]:
# Best treshold
results[index,0]
threshold = compute_thresholds(X_train_full, results[index,0])

In [23]:
#we retrain the model using all the available data so that the model is trained using all the available data

X_train_full_bool = booleanise(X_train_full, threshold)
X_test_bool = booleanise(X_test, threshold)
tm = MultiClassTsetlinMachine(number_of_clauses=1000, T=50, s=5.0)
tm.fit(X_train_full_bool, y_train_full, epochs=100)
# test the model
predictions = tm.predict(X_test_bool)
print("Accuracy:", accuracy_score(y_test, predictions))


Accuracy: 0.9755555555555555
